# 04 · Price model with conformal intervals (CPU is fine; TPU optional via JAX/Keras)

Gradient boosting on log price + **split-conformal** calibration → a distribution-free 90% interval. Exports
`price_model.joblib` with the keys the backend expects (`model`, `q`, `metrics`). Swap the synthetic CSV for your
real quote history (same columns) to make it yours.

In [ ]:
# Get the repo, regenerate the synthetic corpus (deterministic, ~5 s) and fetch the public datasets (checksummed)
!git clone -q https://github.com/<you>/aeronexus-ai.git /content/aeronexus-ai || true
%cd /content/aeronexus-ai/backend
!pip -q install -r requirements.txt
!python scripts/generate_data.py --train 4000
!python scripts/fetch_public_data.py

In [ ]:
!pip -q install lightgbm joblib

In [ ]:
import numpy as np, pandas as pd, math, json, joblib, sys
sys.path.insert(0, ".")
from aeronexus.modules.pricing import featurize, CLASSES, NUMERIC
df = pd.read_csv("data/train/historical_quotes.csv")
X = np.array([featurize(r) for r in df.to_dict("records")]); y = np.log(df["unit_price"].values)
rng = np.random.default_rng(7); idx = rng.permutation(len(df))
tr, cal, te = np.split(idx, [int(.6*len(df)), int(.8*len(df))])

In [ ]:
import lightgbm as lgb
from sklearn.ensemble import GradientBoostingRegressor
cands = {
  "lightgbm": lgb.LGBMRegressor(n_estimators=600, learning_rate=0.03, num_leaves=31, subsample=0.9, colsample_bytree=0.9, verbose=-1),
  "sk_gbr": GradientBoostingRegressor(n_estimators=250, max_depth=3, learning_rate=0.08, subsample=0.9, random_state=7),
}
scores = {}
for name, m in cands.items():
    m.fit(X[tr], y[tr]); p = m.predict(X[cal])
    scores[name] = float(np.mean(np.abs(np.exp(p) - np.exp(y[cal])) / np.exp(y[cal])))
best = min(scores, key=scores.get); model = cands[best]; print(scores, "→", best)

In [ ]:
alpha = 0.10
res = np.abs(model.predict(X[cal]) - y[cal])
q = float(np.sort(res)[min(len(res)-1, math.ceil((len(cal)+1)*(1-alpha)) - 1)])
pt = model.predict(X[te])
metrics = {"model": best, "test_mape": float(np.mean(np.abs(np.exp(pt)-np.exp(y[te]))/np.exp(y[te]))),
           "test_coverage": float(np.mean(np.abs(pt - y[te]) <= q)), "target_coverage": 1 - alpha,
           "n_train": len(tr), "n_cal": len(cal), "n_test": len(te)}
print(json.dumps(metrics, indent=1))
assert metrics["test_coverage"] >= 0.85, "intervals under-cover: recalibrate before shipping"
joblib.dump({"model": model, "q": q, "metrics": metrics}, "/content/price_model.joblib")
ARTIFACTS = ["/content/price_model.joblib"]

In [ ]:
# Save artifacts to Google Drive (survives the Colab session; avoids flaky browser downloads of big files).
# In VS Code: python scripts/import_models.py <the downloaded aeronexus_export.zip or Drive folder>
import hashlib, json, os, platform, shutil, sys, time
from google.colab import drive
drive.mount("/content/drive")
OUT = "/content/drive/MyDrive/aeronexus/export"
os.makedirs(OUT, exist_ok=True)
for item in ARTIFACTS:                                   # files or folders produced above
    dst = os.path.join(OUT, os.path.basename(item.rstrip("/")))
    (shutil.copytree(item, dst, dirs_exist_ok=True) if os.path.isdir(item) else shutil.copy2(item, dst))
def sha(p):
    h = hashlib.sha256(); h.update(open(p, "rb").read()); return h.hexdigest()
import sklearn, numpy
man_path = os.path.join(OUT, "manifest.json")
man = json.load(open(man_path)) if os.path.exists(man_path) else {"artifacts": {}}
for root, _, fs in os.walk(OUT):
    for f in fs:
        if f != "manifest.json":
            p = os.path.join(root, f); man["artifacts"][os.path.relpath(p, OUT)] = sha(p)
man.update({"exported_at": time.strftime("%Y-%m-%dT%H:%M:%S"), "python": platform.python_version(),
            "versions": {"scikit-learn": sklearn.__version__, "numpy": numpy.__version__}})
json.dump(man, open(man_path, "w"), indent=1)
shutil.make_archive("/content/drive/MyDrive/aeronexus/aeronexus_export", "zip", OUT)
print("saved to Drive: MyDrive/aeronexus/aeronexus_export.zip  →", sorted(man["artifacts"])[:12])